# Exploration of LAUS Unemployment Data
 
Data documentation notes available online at: https://download.bls.gov/pub/time.series/la/la.txt
Notes from the documentation:
- "Rates and ratios are expressed as percents with one
decimal place."
- location, survey series both in the series_id
    - la.area file contains the location codes
    - la.series file contains the series codes I *think*


Important codes:
- Unemployment rate measure code: 03
- area_type for counties and equivalents: F
- 




In [151]:
# SETUP
import pandas as pd

laus = pd.read_csv(
    "../data_raw/LAUS/la.data.64.County",
    sep="\t"
)

area = pd.read_csv(
    "../data_raw/LAUS/la.area",
    sep="\t"
)

series = pd.read_csv(
    "../data_raw/LAUS/la.series",
    sep="\t"
)

measure = pd.read_csv(
    "../data_raw/LAUS/la.measure",
    sep="\t"
)

area_type = pd.read_csv(
    "../data_raw/LAUS/la.area_type",
    sep="\t"
)

## Main Data Overview + Pre-Clean

### Note:

During my exploratory analysis, I parsed the series_id field directly to better understand the structure of the LAUS identifiers and verify how survey, area, and measure information are encoded. After reviewing the full LAUS documentation in more detail, I realized the la.series and la.area files provide this metadata directly and are meant to be merged with the main data files for this exact purpose. I will be using the offical metadata files in my official cleaning file, but I kept this here to document my exploratory process!

In [152]:
# Checking dataset format
print(laus.head())
print('#################')
print(laus.columns)

# Cleaning up column names
print('#################')
laus.columns = laus.columns.str.strip()


   series_id                       year period         value footnote_codes
0  LAUCN010010000000003            1990    M01           6.5            NaN
1  LAUCN010010000000003            1990    M02           6.4            NaN
2  LAUCN010010000000003            1990    M03           5.6            NaN
3  LAUCN010010000000003            1990    M04           6.6            NaN
4  LAUCN010010000000003            1990    M05           6.1            NaN
#################
Index(['series_id                     ', 'year', 'period', '       value',
       'footnote_codes'],
      dtype='str')
#################


In [153]:
laus_2 = laus.copy()
# Separating series_id into its respective components according to documentation
laus_2['survey'] = laus_2['series_id'].str[:2]
laus_2['adjusted'] = laus_2['series_id'].str[2:3]
laus_2['area_code'] = laus_2['series_id'].str[3:18]
laus_2['measure'] = laus_2['series_id'].str[18:]

print('#################################')
print(laus_2.head())

#################################
                        series_id  year period         value footnote_codes  \
0  LAUCN010010000000003            1990    M01           6.5            NaN   
1  LAUCN010010000000003            1990    M02           6.4            NaN   
2  LAUCN010010000000003            1990    M03           5.6            NaN   
3  LAUCN010010000000003            1990    M04           6.6            NaN   
4  LAUCN010010000000003            1990    M05           6.1            NaN   

  survey adjusted        area_code       measure  
0     LA        U  CN0100100000000  03            
1     LA        U  CN0100100000000  03            
2     LA        U  CN0100100000000  03            
3     LA        U  CN0100100000000  03            
4     LA        U  CN0100100000000  03            


## Preparing and Merging Metadata Files

In [ ]:
# AREA FILE PREPARATION

# File cleanup
print(area.columns) # all set

# Restricting to county and equivalent codes
area = area[area['area_type_code'] == 'F']

# Separating county and state to use when merging geographic index in
area[['county', 'state']] = area['area_text'].str.split(', ', expand=True)


print(area.head())

Index(['area_type_code', 'area_code', 'area_text', 'display_level',
       'selectable', 'sort_sequence'],
      dtype='str')
     area_type_code        area_code           area_text  display_level  \
1208              F  CN0100100000000  Autauga County, AL              0   
1209              F  CN0100300000000  Baldwin County, AL              0   
1210              F  CN0100500000000  Barbour County, AL              0   
1211              F  CN0100700000000     Bibb County, AL              0   
1212              F  CN0100900000000   Blount County, AL              0   

     selectable  sort_sequence          county state  
1208          T             33  Autauga County    AL  
1209          T             34  Baldwin County    AL  
1210          T             35  Barbour County    AL  
1211          T             36     Bibb County    AL  
1212          T             37   Blount County    AL  


In [ ]:
# SERIES FILE PREPARATION

# Cleaning up file columns
series.columns = series.columns.str.strip()

# Restricting to county/county-equivalent subset
series = series[series['area_type_code'] == 'F']
print(series.head())

Index(['series_id                     ', 'area_type_code', 'area_code',
       'measure_code', 'seasonal', 'srd_code', 'series_title',
       'footnote_codes', 'begin_year', 'begin_period', 'end_year',
       'end_period'],
      dtype='str')
                           series_id area_type_code        area_code  \
1186  LAUCN010010000000003                        F  CN0100100000000   
1187  LAUCN010010000000004                        F  CN0100100000000   
1188  LAUCN010010000000005                        F  CN0100100000000   
1189  LAUCN010010000000006                        F  CN0100100000000   
1190  LAUCN010030000000003                        F  CN0100300000000   

      measure_code seasonal  srd_code  \
1186             3        U         1   
1187             4        U         1   
1188             5        U         1   
1189             6        U         1   
1190             3        U         1   

                                   series_title footnote_codes  begin_year  \

In [157]:
# Merging metadata files
laus = pd.merge(laus, series, on=('series_id'))